# Traffic Congestion Prediction Model

## Goal
The goal of this section is to train a machine learning model that can predict traffic speed based on engineered features.

This is an important step because:
- Traffic speed determines how congested a road is
- If we can predict speed, we can estimate travel time
- This allows us to choose the most efficient route

---

## What is a Model?

A model is a function that learns patterns from data and uses those patterns to make predictions.

In this project:
- Input (X): Features about the road, time, and traffic patterns
- Output (y): Traffic speed

The model learns:
"Given these conditions, what will the traffic speed be?"

---

## Datasets Used

We are primarily using the `congestion_ml` dataset for modeling.

### congestion_ml
This dataset contains:
- Engineered features (time, lag, rolling averages)
- Traffic-related metrics (volume, speed, travel time)
- Encoded categorical variables (borough, direction)

This is the dataset we use to TRAIN the model.

---

### routing_edges
This dataset represents the road network:
- Nodes (intersections)
- Edges (roads between nodes)
- Travel time and distance

This will be used later for routing.

---

### routing_nodes
This dataset contains:
- Node IDs
- Latitude and longitude

This helps map the road network spatially.

---

## Important
For this notebook, we only use `congestion_ml` for training the model.

In [37]:
# imports

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

In [38]:
## load dataset

df = pd.read_csv("../../data/ml_datasets/congestion_ml.csv")
print(f"Rows and columns before cleaning {df.shape}")
df.head()


print(df.isnull().sum())


Rows and columns before cleaning (123334, 34)
SegmentID                0
lat                      0
lon                      0
avg_speed                0
avg_travel_time          0
avg_vol_hist             0
peak_vol_hist            0
std_vol_hist             0
peak_hour_hist           0
hour                     0
day_of_week              0
is_weekend               0
is_rush_hour             0
hour_sin                 0
hour_cos                 0
month_sin                0
month_cos                0
dow_sin                  0
dow_cos                  0
vol_lag_1                0
vol_lag_2                0
vol_lag_3                0
vol_rolling_avg_3        0
borough_Bronx            0
borough_Brooklyn         0
borough_Manhattan        0
borough_Queens           0
borough_Staten Island    0
Direction_EB             0
Direction_NB             0
Direction_SB             0
Direction_WB             0
Vol                      0
is_congested             0
dtype: int64


## Data Cleaning

Before training any models, we need to ensure the dataset is clean.

Feature engineering (such as lag and rolling features) often creates missing values (NaN). These occur because some rows do not have enough historical data.

We remove these rows so the model only learns from complete and valid data.

In [39]:
#Data before Clean
print("Shape before cleaning:",df.shape,"\n")
print("Missing values:")
print(df.isnull().sum())

Shape before cleaning: (123334, 34) 

Missing values:
SegmentID                0
lat                      0
lon                      0
avg_speed                0
avg_travel_time          0
avg_vol_hist             0
peak_vol_hist            0
std_vol_hist             0
peak_hour_hist           0
hour                     0
day_of_week              0
is_weekend               0
is_rush_hour             0
hour_sin                 0
hour_cos                 0
month_sin                0
month_cos                0
dow_sin                  0
dow_cos                  0
vol_lag_1                0
vol_lag_2                0
vol_lag_3                0
vol_rolling_avg_3        0
borough_Bronx            0
borough_Brooklyn         0
borough_Manhattan        0
borough_Queens           0
borough_Staten Island    0
Direction_EB             0
Direction_NB             0
Direction_SB             0
Direction_WB             0
Vol                      0
is_congested             0
dtype: int64


In [40]:
#Data Clean
df = df.dropna(axis=1, how="all")# drops columns where ALL values are NaN
print("Shape after cleaning:", df.shape)
print(df.isnull().sum())
df.head()


Shape after cleaning: (123334, 34)
SegmentID                0
lat                      0
lon                      0
avg_speed                0
avg_travel_time          0
avg_vol_hist             0
peak_vol_hist            0
std_vol_hist             0
peak_hour_hist           0
hour                     0
day_of_week              0
is_weekend               0
is_rush_hour             0
hour_sin                 0
hour_cos                 0
month_sin                0
month_cos                0
dow_sin                  0
dow_cos                  0
vol_lag_1                0
vol_lag_2                0
vol_lag_3                0
vol_rolling_avg_3        0
borough_Bronx            0
borough_Brooklyn         0
borough_Manhattan        0
borough_Queens           0
borough_Staten Island    0
Direction_EB             0
Direction_NB             0
Direction_SB             0
Direction_WB             0
Vol                      0
is_congested             0
dtype: int64


,SegmentID,lat,lon,avg_speed,avg_travel_time,avg_vol_hist,peak_vol_hist,std_vol_hist,peak_hour_hist,hour,...,borough_Brooklyn,borough_Manhattan,borough_Queens,borough_Staten Island,Direction_EB,Direction_NB,Direction_SB,Direction_WB,Vol,is_congested
0,-0.258833,-1.378228,-3.195527,1.689452,-0.567263,-0.604429,-0.443698,-0.518378,-2.255487,-1.652075,...,False,False,False,True,False,False,False,True,16.0,0
1,-0.258833,-1.378228,-3.195527,1.689452,-0.567263,-0.604429,-0.443698,-0.518378,-2.255487,-1.652075,...,False,False,False,True,False,False,False,True,19.0,0
2,-0.258833,-1.378228,-3.195527,1.689452,-0.567263,-0.604429,-0.443698,-0.518378,-2.255487,-1.652075,...,False,False,False,True,False,False,False,True,10.0,0
3,-0.258833,-1.378228,-3.195527,1.689452,-0.567263,-0.604429,-0.443698,-0.518378,-2.255487,-1.652075,...,False,False,False,True,False,False,False,True,12.0,0
4,-0.258833,-1.378228,-3.195527,1.689452,-0.567263,-0.604429,-0.443698,-0.518378,-2.255487,-1.507990,...,False,False,False,True,False,False,False,True,16.0,0


## Define Features (X) and Target (y)

We split the dataset into:

- Features (X): all columns used to make predictions  
- Target (y): the value we want to predict  

In this project, we predict **traffic speed**.

We remove the target column from X to prevent data leakage.

In [ ]:
# Feature Selection Mode (Choose ONE)

USE_LEAKAGE_SAFE_MODE = True  # change to False if you want simple version

if USE_LEAKAGE_SAFE_MODE:
    # clean ML version (recommended)
    drop_cols = [
        "avg_speed",
        "is_congested",
        "avg_travel_time",
        "Vol",
        # Lag/rolling features are shifted copies of Vol (the target).
        # Keeping them causes the model to learn Vol ≈ vol_lag_1 rather
        # than learning from genuine time and location signals.
        "vol_lag_1",
        "vol_lag_2",
        "vol_lag_3",
        "vol_rolling_avg_3",
    ]
    print("Using CLEAN MODE (recommended)")
else:
    # simple baseline version
    drop_cols = ["avg_speed", "is_congested", "Vol",
                 "vol_lag_1", "vol_lag_2", "vol_lag_3", "vol_rolling_avg_3"]
    print("Using SIMPLE MODE")

X = df.drop(columns=drop_cols)
y = df[["Vol"]]  # multi-output: predict volume and speed

print("X shape:", X.shape)
print("y shape:", y.shape)

# dropping SegmentID so model learns from real signals
X = X.drop(columns=["SegmentID"])


Using CLEAN MODE (recommended)
X shape: (123334, 26)
y shape: (123334, 2)


## Train/Test Split

We divide the data into:

- Training set (80%) → used to train the model  
- Testing set (20%) → used to evaluate the model  

This allows us to test how well the model performs on unseen data.

In [42]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Model 1: Linear Regression

Linear Regression is the simplest machine learning model.

It assumes that the relationship between features and the target is linear.

In this context, it tries to learn relationships like:
- higher traffic volume → lower speed  
- rush hour → slower traffic  

This model is useful as a baseline to compare more complex models against.

In [43]:
import matplotlib.pyplot as plt

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)

targets = ["Vol", "avg_speed"]

#--------Linear Regression Model-----------
for i, target in enumerate(targets):
    sse  = np.sum((y_test.iloc[:, i].values - lr_pred[:, i]) ** 2)
    mae  = mean_absolute_error(y_test.iloc[:, i], lr_pred[:, i])
    rmse = np.sqrt(mean_squared_error(y_test.iloc[:, i], lr_pred[:, i]))
    print(f"[{target}] SSE: {sse:.2f} | MAE: {mae:.4f} | RMSE: {rmse:.4f}")

# Scatter plots — one per target
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for i, (ax, target) in enumerate(zip(axes, targets)):
    actual = y_test.iloc[:, i]
    pred   = lr_pred[:, i]
    ax.scatter(actual, pred, alpha=0.5, s=20)
    ax.plot([actual.min(), actual.max()],
            [actual.min(), actual.max()],
            linestyle="--", linewidth=2)
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")
    ax.set_title(f"Linear Regression: {target}")
    ax.grid(True)
plt.tight_layout()
plt.show()

# Coefficients — one row per target
for i, target in enumerate(targets):
    coef_df = pd.DataFrame({"Feature": X.columns, "Coefficient": lr_model.coef_[i]})
    print(f"\nCoefficients for {target}:")
    print(coef_df.sort_values(by="Coefficient", ascending=False).to_string())


## Model 2: Decision Tree

A Decision Tree model splits the data into branches based on feature values.

It learns rules such as:
- if rush hour → predict lower speed  
- if weekend → predict higher speed  

This allows the model to capture non-linear patterns in traffic behavior.

In [ ]:
from sklearn.tree import plot_tree

dt_model = DecisionTreeRegressor(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)

dt_pred = dt_model.predict(X_test)

targets = ["Vol", "avg_speed"]

#--------Decision Tree Model-----------
for i, target in enumerate(targets):
    sse  = np.sum((y_test.iloc[:, i].values - dt_pred[:, i]) ** 2)
    mae  = mean_absolute_error(y_test.iloc[:, i], dt_pred[:, i])
    rmse = np.sqrt(mean_squared_error(y_test.iloc[:, i], dt_pred[:, i]))
    print(f"[{target}] SSE: {sse:.2f} | MAE: {mae:.4f} | RMSE: {rmse:.4f}")

# Decision Tree visualization
plt.figure(figsize=(20, 10))
plot_tree(
    dt_model,
    feature_names=X.columns,
    filled=True,
    rounded=True,
    fontsize=10,
    max_depth=3
)
plt.title("Decision Tree Visualization (Top Levels)")
plt.show()

# Scatter plots — one per target
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for i, (ax, target) in enumerate(zip(axes, targets)):
    actual = y_test.iloc[:, i]
    pred   = dt_pred[:, i]
    ax.scatter(actual, pred, alpha=0.5, s=20)
    ax.plot([actual.min(), actual.max()],
            [actual.min(), actual.max()],
            linestyle='--', linewidth=2)
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")
    ax.set_title(f"Decision Tree: {target}")
    ax.grid(True)
plt.tight_layout()
plt.show()

# Feature importance
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": dt_model.feature_importances_
})
print(importance.sort_values("Importance", ascending=False).to_string())


## Model Evaluation

We evaluate each model using:

- MAE (Mean Absolute Error): average difference between predicted and actual values  
- RMSE (Root Mean Squared Error): penalizes larger errors more  

Lower values indicate better performance.

In [ ]:
def evaluate(y_true, y_pred, targets):
    results = {}
    for i, target in enumerate(targets):
        mae  = mean_absolute_error(y_true.iloc[:, i], y_pred[:, i])
        rmse = np.sqrt(mean_squared_error(y_true.iloc[:, i], y_pred[:, i]))
        results[target] = (mae, rmse)
    return results

targets = ["Vol", "avg_speed"]

lr_results = evaluate(y_test, lr_pred, targets)
dt_results = evaluate(y_test, dt_pred, targets)

print("Linear Regression:")
for target, (mae, rmse) in lr_results.items():
    print(f"  {target} -> MAE: {mae:.4f}, RMSE: {rmse:.4f}")

print("\nDecision Tree:")
for target, (mae, rmse) in dt_results.items():
    print(f"  {target} -> MAE: {mae:.4f}, RMSE: {rmse:.4f}")


## Model Comparison

We compare all models to determine which performs best.

The best model is the one with the lowest error.

This model will later be used in the routing algorithm to predict traffic speed and estimate travel time.

In [ ]:
rows = []
for target in targets:
    rows.append({
        "Target":  target,
        "LR MAE":  lr_results[target][0],
        "LR RMSE": lr_results[target][1],
        "DT MAE":  dt_results[target][0],
        "DT RMSE": dt_results[target][1],
    })
pd.DataFrame(rows).set_index("Target")


## Final Interpretation

- Which model performed best?
- How much better was it compared to the others?
- Why might this model perform better?

In general:
- Linear Regression is simple but limited  
- Decision Trees capture more complex patterns  
- Random Forest combines multiple trees and often achieves the best performance  

The selected model will be used to predict traffic speeds, which will directly impact route optimization.

Final Interpretation:

The Decision Tree model performed better than Linear Regression.

- Linear Regression:
  MAE ≈ 0.7004, RMSE ≈ 0.8395

- Decision Tree:
  MAE ≈ 0.3115, RMSE ≈ 0.5011

The Decision Tree reduced error significantly (over 50% lower MAE), meaning its predictions are much closer to the actual traffic speeds.

This happens because traffic behavior is not purely linear. Factors such as location (latitude/longitude), traffic history, and time-related features create complex, non-linear relationships.

Linear Regression assumes a straight-line relationship, so it cannot capture these patterns well.

In contrast, the Decision Tree splits the data into regions based on feature values, allowing it to model different traffic behaviors in different conditions.

Feature importance shows that location (latitude and longitude) plays a major role in predictions, indicating that traffic speed varies significantly across different areas.

Therefore, the Decision Tree is a better model for predicting traffic speed in this dataset.

